# Embeddings erstellen

Ein **Word Embedding** ist ein dichter Vektor, der ein Wort repräsentiert und so gelernt wird, dass Wörter, die in ähnlichen Kontexten verwendet werden, am Ende ähnliche Vektoren haben. Das ist nützlicher als One-Hot Encoding, bei dem jedes Wort ein dünnbesetzter Vektor ist, der von allen anderen gleich weit entfernt ist und keinerlei Ähnlichkeit abbildet.

In diesem Notebook trainieren wir ein winziges Embedding-Modell von Grund auf mit einer Handvoll Sätzen, damit der ganze Mechanismus sichtbar wird. Wir machen aus Wörtern Kontextpaare, trainieren ein neuronales Netz mit einer verborgenen Schicht darauf, Kontextwörter vorherzusagen, und lesen dann die Gewichte der verborgenen Schicht als Embeddings aus.

## Lernziele

Am Ende dieses Notebooks kannst du:

- aus unverarbeiteten Sätzen Trainingspaare von Ziel- und Kontextwörtern bilden.
- ein neuronales Netz mit einer verborgenen Schicht erstellen und trainieren, das Word Embeddings lernt.
- die gelernten Embeddings aus den Gewichten des Netzes extrahieren.
- die Embeddings in zwei Dimensionen visualisieren und die Anordnung interpretieren.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# Embeddings zeichnen
import matplotlib.pyplot as plt

# Deep Learning:
from keras.models import Sequential
from keras.layers import Dense, Input

### Eigene Funktionen laden

In [ ]:
import sys

# Diese Lösung liegt in solutions/, eine Ebene unter dem Stammverzeichnis des Repositorys,
# daher fügen wir den übergeordneten Ordner zum Pfad hinzu, um das gemeinsame Hilfspaket
# extra/ zu importieren.
sys.path.append("..")

# Wir laden eigene Funktionen aus einer Python-Datei
from extra.utility import text_preprocessing, create_unique_word_dict

### Daten einlesen

In [ ]:
# Text aus dem Eingabeordner einlesen
texts = pd.read_csv("../data/sample.csv")
texts = [x for x in texts["text"]]
texts

## Trainingsdaten erstellen

Die Idee hinter diesen Embeddings ist der **Skip-Gram**-Ansatz: Ausgehend von jedem Zielwort versuchen wir, die Wörter vorherzusagen, die in seiner Nähe stehen. Wir lassen ein Fenster der Größe 2 über jeden Satz gleiten und halten jedes Paar (Ziel, Kontext) fest. Das Modell, das lernt, aus einem Zielwort den Kontext vorherzusagen, ist gezwungen, verwandten Wörtern ähnliche interne Repräsentationen zu geben, und diese Repräsentationen sind die Embeddings.

Wir setzen `window=2`, um Paare aus 2 Wörtern zu bilden, die im Datensatz zusammen vorkommen.

In [ ]:
# Fenster für den Kontext festlegen
window = 2

# Platzhalter für das Durchlaufen der Wortliste anlegen
word_lists = []
all_text = []

- Wir gehen alle Sätze im Datensatz durch und verarbeiten sie vor (zerlegen sie in Tokens und entfernen die Stoppwörter)
- Dann bilden wir Paare aus Wörtern, die innerhalb einer Fenstergröße von 2 nahe beieinander stehen.
  - Für zwei Wörter, etwa `a` und `b`, werden die Paare `a,b` und `b,a` gebildet

In [ ]:
for text in texts:
    # Text bereinigen
    text = text_preprocessing(text)

    # An die Liste mit allen Texten anhängen
    all_text += text

    # Kontext-Dictionary erstellen
    for i, word in enumerate(text):
        for w in range(window):
            # Kontext der folgenden *window* Wörter abrufen
            if i + 1 + w < len(text):
                word_lists.append([word] + [text[(i + 1 + w)]])
            # Kontext der vorangehenden *window* Wörter abrufen
            if i - w - 1 >= 0:
                word_lists.append([word] + [text[(i - w - 1)]])

In [ ]:
word_lists

Wir holen alle verschiedenen Wörter aus unserem Datensatz heraus

In [ ]:
unique_word_dict = create_unique_word_dict(all_text)

In [ ]:
unique_word_dict

In [ ]:
# Anzahl der Features (verschiedene Wörter) festlegen
n_words = len(unique_word_dict)

# Alle verschiedenen Wörter abrufen
words = list(unique_word_dict.keys())

In [ ]:
# Matrizen X und Y per One-Hot Encoding erstellen
X = []
Y = []

for i, word_list in tqdm(enumerate(word_lists)):
    # Indizes abrufen
    main_word_index = unique_word_dict.get(word_list[0])
    context_word_index = unique_word_dict.get(word_list[1])

    # Platzhalter anlegen
    X_row = np.zeros(n_words)
    Y_row = np.zeros(n_words)

    # Zielwort per One-Hot Encoding kodieren
    X_row[main_word_index] = 1

    # Wörter der Y-Matrix per One-Hot Encoding kodieren
    Y_row[context_word_index] = 1

    # An die Hauptmatrizen anhängen
    X.append(X_row)
    Y.append(Y_row)

In [ ]:
# Listen in NumPy-Arrays umwandeln
X = np.asarray(X)
Y = np.asarray(Y)

## Embedding-Modell trainieren

Wir erstellen ein Netz mit einer verborgenen Schicht. Die Eingabe ist ein One-Hot-Vektor über das Vokabular, die verborgene Schicht hat nur 2 Neuronen (diese Größe ist die Embedding-Dimension), und die Ausgabe ist eine Softmax über das Vokabular, die Kontextwörter vorhersagt. Nach dem Training verwerfen wir die Ausgabeschicht: Die Gewichte der verborgenen Schicht sind die Wortvektoren, die wir suchen.

In [ ]:
# Embedding-Größe festlegen
embed_size = 2

Wir erstellen ein neuronales Netz mit einer Schicht, mit `input dimension = total number of unique words`, was unser Vokabular repräsentiert. Außerdem gilt `output dimension = size of vocabulary`, um für jedes Wort Embeddings derselben Dimension zu erhalten.

In [ ]:
# Neuronales Netz definieren
model = Sequential()

# Schichten
model.add(Input(shape=(X.shape[1],)))
model.add(Dense(units=embed_size, activation="linear"))
model.add(Dense(units=Y.shape[1], activation="softmax"))


# ANN kompilieren
model.compile(optimizer="adam", loss="categorical_crossentropy")
print(model.summary())

In [ ]:
# Gewichte des Netzes optimieren
model.fit(x=X, y=Y, batch_size=256, epochs=1000)

**Ergebnis:** Die Beispieldaten enthalten 12 kurze Sätze mit 21 verschiedenen Wörtern, die 84 Ziel-Kontext-Paare ergeben. Weil die verborgene Schicht 2 Neuronen hat, wird jedes der 21 Wörter nach dem Training durch einen 2D-Vektor repräsentiert.

Die Vorhersagen des Netzes nutzen wir nicht direkt. Das Trainingsziel ist nur ein Mittel, um verwandte Wörter (zum Beispiel `king`, `queen`, `prince`, `princess`) im 2D-Raum an ähnliche Positionen zu schieben, und genau das zeigt das Streudiagramm unten. Bei so wenigen Daten und einer zufälligen Initialisierung ändert sich die genaue Anordnung von Durchlauf zu Durchlauf, aber Wörter mit gemeinsamen Kontexten liegen meist nahe beieinander.

Nachdem wir das Netz trainiert haben, extrahieren wir die trainierten Gewichte aus der ersten Schicht des neuronalen Netzes: Sie sind die Embeddings für unser Vokabular

In [ ]:
# Die Eingabeschicht
weights = model.get_weights()[0]

In [ ]:
# Dictionary zum Speichern der Embeddings erstellen. Der Schlüssel ist jeweils eines
# der verschiedenen Wörter, der Wert der numerische Vektor
embedding_dict = {}
for word in words:
    embedding_dict.update({word: weights[unique_word_dict.get(word)]})

## Embeddings visualisieren

### 2D-Visualisierung

In [ ]:
# Embeddings darstellen
plt.figure(figsize=(10, 10))
for word in list(unique_word_dict.keys()):
    coord = embedding_dict.get(word)
    plt.scatter(coord[0], coord[1])
    plt.annotate(word, (coord[0], coord[1]))

### 3D-Visualisierung

Wir können die Embeddings auch in 3D ansehen. Setze `embed_size = 3` in der Zelle oben und führe das Notebook erneut aus, dann zeichnet die Zelle unten mit Plotly ein interaktives 3D-Streudiagramm.

Ziehe am Plot, um ihn zu drehen, und fahre mit der Maus über einen beliebigen Punkt, um sein Wort und seine genauen Koordinaten abzulesen. Das löst das Hauptproblem eines statischen 3D-Bildes, bei dem sich die Tiefe nicht beurteilen lässt und du nicht erkennen kannst, wo ein Punkt wie `man` tatsächlich liegt.

In [ ]:
# Interaktiver 3D-Plot (braucht embed_size = 3 oben, dann das Notebook erneut ausführen)
if embed_size >= 3:
    import plotly.express as px

    words = list(embedding_dict.keys())
    coords = np.array([embedding_dict[w] for w in words])

    fig = px.scatter_3d(x=coords[:, 0], y=coords[:, 1], z=coords[:, 2], text=words)
    fig.update_traces(
        hovertemplate="%{text}<br>x=%{x:.2f} y=%{y:.2f} z=%{z:.2f}<extra></extra>"
    )
    fig.update_layout(
        title="Word embeddings in 3D (drag to rotate, hover for position)"
    )
    fig.show()
else:
    print(
        "Set embed_size = 3 in the cell above and rerun the notebook to see the interactive 3D plot."
    )

### Embeddings speichern

Die Zelle unten schreibt die Embeddings in lesbarer Form (`word: [vector]`) in `embeddings_output/embedding.txt`. Das ist praktisch für einen schnellen Blick, aber kein Format, das irgendein Tool erwartet.

> Der interaktive [TensorFlow Embedding Projector](http://projector.tensorflow.org/) braucht stattdessen zwei tabulatorgetrennte Dateien: eine `vecs.tsv` mit den reinen Vektoren und eine `meta.tsv` mit den zugehörigen Wörtern. Diese erzeugst du im optionalen Notebook 5, indem du dem Tutorial dort folgst. Anders gesagt: Diese `.txt`-Ausgabe und die `.tsv`-Dateien des Projectors sind zwei verschiedene Dinge, die `.txt` hier ist also nicht zum Hochladen in den Projector gedacht.

In [ ]:
# Embedding-Vektor in einer txt-Datei speichern
try:
    os.mkdir(f"{os.getcwd()}//embeddings_output")
except Exception as e:
    print(f"Cannot create output folder: {e}")

In [ ]:
with open(f"{os.getcwd()}//embeddings_output//embedding.txt", "w") as f:
    for key, value in embedding_dict.items():
        try:
            f.write(f"{key}: {value}\n")
        except Exception as e:
            print(f"Cannot write word {key} to dict: {e}")

- Diese Embeddings werden später als Embedding-Schicht in größeren neuronalen Netzen wiederverwendet, die für NLP-Aufgaben trainiert werden.
- Eine höhere Embedding-Größe (ein längerer Vektor pro Wort) kann mehr Nuancen erfassen, kostet aber mehr Parameter, die trainiert werden müssen.
- Probiere eine Embedding-Größe von 3 und eine Darstellung in 3D aus, oder bleib beim 2D-Streudiagramm, und achte auf die Unterschiede.

### Du bist dran: die Ähnlichkeit von Wörtern messen

Embeddings platzieren verwandte Wörter nahe beieinander. Quantifiziere das mit der Kosinus-Ähnlichkeit und probiere dann die klassische Analogie aus.

**Schritte**
1. Schreibe mit NumPy `cosine_similarity(a, b)` für zwei Vektoren (Skalarprodukt geteilt durch das Produkt der Normen).
2. Schreibe `most_similar(word)`, um mithilfe von `embedding_dict` die Wörter des Vokabulars (außer `word`) mit der höchsten Kosinus-Ähnlichkeit zurückzugeben.
3. Probiere `most_similar("king")` aus, bilde dann den Analogievektor `king - man + woman` und finde das Wort, das ihm am nächsten liegt.

**Tipp:** Verwende `np.dot` und `np.linalg.norm`. Für die Analogie bildest du zuerst den Zielvektor und vergleichst ihn dann mit jedem Wortvektor.

**Erwartetes Ergebnis:** Mit nur 12 Sätzen ist das Ergebnis grob, aber verwandte Wörter rund um Königshaus oder Geschlecht sollten in der Rangfolge weit oben stehen.

In [ ]:
def cosine_similarity(a, b):
    # Kosinus-Ähnlichkeit: Skalarprodukt geteilt durch das Produkt der Normen
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def most_similar(word, top_n=3):
    # Alle anderen Wörter nach ihrer Kosinus-Ähnlichkeit zu `word` in eine Rangfolge bringen
    target = embedding_dict[word]
    scored = [
        (other, cosine_similarity(target, vec))
        for other, vec in embedding_dict.items()
        if other != word
    ]
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return scored[:top_n]


print("Most similar to 'king':", most_similar("king"))

# Den Analogievektor king - man + woman bilden und das Wort finden, das ihm am nächsten liegt
analogy_vec = embedding_dict["king"] - embedding_dict["man"] + embedding_dict["woman"]
closest = max(
    ((w, cosine_similarity(analogy_vec, v)) for w, v in embedding_dict.items()),
    key=lambda pair: pair[1],
)
print("king - man + woman is closest to:", closest)

**Erklärung**

Die Kosinus-Ähnlichkeit vergleicht die Richtung zweier Vektoren und ignoriert ihre Länge. Das ist die Standardmethode, um zu messen, wie verwandt zwei Embeddings sind. most_similar bringt jedes andere Wort im Vergleich mit dem Zielwort in eine Rangfolge, und die Analogie king - man + woman bildet einen neuen Vektor, dessen nächstgelegenes Wort die Beziehung zwischen den Geschlechtern erfasst. Mit nur 12 Sätzen ist das Ergebnis verrauscht, aber genau diese Methode nutzen große Embedding-Modelle im großen Stil.


### Du bist dran: Fenster und Embedding-Größe untersuchen

Die Fenstergröße und die Embedding-Dimension beeinflussen, was das Modell lernt. Experimentiere und beobachte.

**Schritte**
1. Ändere `window` in der Zelle weiter oben auf `3` und führe das Notebook ab dort erneut aus.
2. Vergleiche die Anzahl der Kontextpaare (`len(word_lists)`) mit dem Wert, den du mit `window = 2` hattest.
3. Setze `embed_size = 3`, führe das Notebook erneut aus und sieh dir den interaktiven 3D-Plot im Abschnitt zur Visualisierung an.

**Tipp:** Die Anzahl der Paare wächst, je breiter das Fenster wird, weil jedes Wort jetzt mehr Nachbarn sieht.

**Erwartetes Ergebnis:** Ein größeres Fenster erzeugt mehr Trainingspaare; eine höhere Embedding-Größe gibt jedem Wort einen längeren Vektor und ergibt einen reichhaltigeren (aber schwerer darstellbaren) Raum.

In [ ]:
# Nachdem du `window` oben geändert und erneut ausgeführt hast, prüfe hier die Anzahl der Kontextpaare.
print(f"window = {window}, number of context pairs = {len(word_lists)}")

**Erklärung**

Ein breiteres Fenster lässt jedes Wort mehr Nachbarn sehen, deshalb wächst die Anzahl der Trainingspaare. Eine höhere Embedding-Größe gibt jedem Wort einen längeren Vektor, der mehr Struktur kodieren kann, kostet aber mehr Parameter und ergibt einen Raum, der zu hochdimensional ist, um ihn direkt darzustellen.


## Zusammenfassung

In diesem Notebook hast du:

- mit einem gleitenden Fenster aus unverarbeiteten Sätzen Ziel-Kontext-Wortpaare gebildet.
- ein neuronales Netz mit einer verborgenen Schicht erstellt und trainiert, das Word Embeddings lernt.
- die Embeddings aus den Gewichten der verborgenen Schicht extrahiert.
- die 21 Wortvektoren in zwei Dimensionen visualisiert und die Anordnung interpretiert.

## Quellen & weiterführende Literatur

- [**The Illustrated Word2vec**](https://jalammar.github.io/illustrated-word2vec/): Eine visuelle Erklärung, wie Word Embeddings gelernt werden und was sie repräsentieren.
- [**Keras: The Sequential model**](https://keras.io/api/models/sequential/): Referenz zur Modell-API, mit der das Netz erstellt wird.